# Treino YOLO11 - Deteccao de EPI (Colab) - v4 (+ Construction-PPE)

Antes de rodar:
1. Suba `yolo_v2.zip` para `MyDrive/TCC/yolo_dataset/yolo_v2.zip` (se ainda nao estiver la).
2. Suba `novos_frames_dataset.zip` para `MyDrive/TCC/yolo_dataset/novos_frames_dataset.zip`.
3. Suba `construction_ppe_dataset.zip` para `MyDrive/TCC/yolo_dataset/construction_ppe_dataset.zip` (dataset publico Ultralytics, 1416 imagens, reforca principalmente gloves e goggles).
4. Em `Runtime > Change runtime type`, selecione GPU.
5. Rode as celulas abaixo em ordem.


In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
ZIP_PATH = '/content/drive/MyDrive/TCC/yolo_dataset/yolo_v2.zip'
DATA_DIR = '/content'

!mkdir -p {DATA_DIR}
!unzip -q -o "{ZIP_PATH}" -d /content

# Precaucao de permissao (o zip da v1 vinha do Windows com permissoes zeradas;
# este foi gerado no Linux, mas o chmod nao faz mal nenhum manter).
!chmod -R u+rwX {DATA_DIR}
!ls -la {DATA_DIR}

In [ ]:
# Checagem antes de gastar GPU: confirma que os arquivos estao realmente visiveis
import os

for split in ['train', 'val', 'test']:
    img_dir = f'{DATA_DIR}/{split}/images'
    lbl_dir = f'{DATA_DIR}/{split}/labels'
    n_img = len(os.listdir(img_dir))
    n_lbl = len(os.listdir(lbl_dir))
    print(f'{split}: {n_img} imagens, {n_lbl} labels')
    assert n_img > 0, f'ERRO: {img_dir} esta vazio ou inacessivel!'
    assert n_img == n_lbl, f'ERRO: numero de imagens e labels difere em {split}!'

print('OK - dataset pronto para treino')

## Mesclar frames reais anotados

Junta as ~100 imagens novas (com oversample 8x = 800 copias) no split de treino existente, e separa as 25 de teste real num diretorio a parte (nunca entra no treino - serve para medir o modelo antes/depois).

In [ ]:
NOVOS_ZIP = '/content/drive/MyDrive/TCC/yolo_dataset/novos_frames_dataset.zip'

!unzip -q -o "{NOVOS_ZIP}" -d /content/novos

import shutil, os

extra_img = '/content/novos/novos_frames_dataset/train_extra/images'
extra_lbl = '/content/novos/novos_frames_dataset/train_extra/labels'
for fname in os.listdir(extra_img):
    shutil.copy(os.path.join(extra_img, fname), f'{DATA_DIR}/train/images/{fname}')
for fname in os.listdir(extra_lbl):
    shutil.copy(os.path.join(extra_lbl, fname), f'{DATA_DIR}/train/labels/{fname}')

REAL_TEST_DIR = '/content/real_world_test'
shutil.copytree('/content/novos/novos_frames_dataset/real_world_test', REAL_TEST_DIR, dirs_exist_ok=True)

print('train agora com', len(os.listdir(f'{DATA_DIR}/train/images')), 'imagens')
print('real_world_test:', len(os.listdir(f'{REAL_TEST_DIR}/images')), 'imagens')


## Mesclar dataset Construction-PPE (Ultralytics)

Dataset publico com 1416 imagens (1132 treino / 143 val / 141 teste), reforcando principalmente `gloves` (+1445 instancias) e `goggles` (+518 instancias) - as duas classes mais fracas ate aqui. Ja vem com split proprio, entao mescla direto em train/val/test sem oversample.

In [ ]:
CPPE_ZIP = '/content/drive/MyDrive/TCC/yolo_dataset/construction_ppe_dataset.zip'

!unzip -q -o "{CPPE_ZIP}" -d /content/cppe

import shutil, os

CPPE_ROOT = '/content/cppe/construction_ppe_raw'
for split in ['train', 'val', 'test']:
    src_img = f'{CPPE_ROOT}/{split}/images'
    src_lbl = f'{CPPE_ROOT}/{split}/labels'
    for fname in os.listdir(src_img):
        shutil.copy(os.path.join(src_img, fname), f'{DATA_DIR}/{split}/images/{fname}')
    for fname in os.listdir(src_lbl):
        shutil.copy(os.path.join(src_lbl, fname), f'{DATA_DIR}/{split}/labels/{fname}')
    print(split, '->', len(os.listdir(f'{DATA_DIR}/{split}/images')), 'imagens no total agora')


In [ ]:
%pip install -q ultralytics
!yolo settings sync=False
import ultralytics
ultralytics.checks()

In [ ]:
# Reescreve data.yaml com caminhos absolutos do Colab
data_yaml = f"""train: {DATA_DIR}/train/images
val: {DATA_DIR}/val/images
test: {DATA_DIR}/test/images
nc: 5
names: ['helmet', 'vest', 'gloves', 'boots', 'goggles']
"""

with open(f'{DATA_DIR}/data.yaml', 'w') as f:
    f.write(data_yaml)

print(data_yaml)

## Treino

Mesmos hiperparametros usados localmente (`runs/detect/train-3/args.yaml`). Os resultados sao salvos direto no Drive (`project=`) para nao se perderem se a sessao do Colab cair.

In [ ]:
!mkdir -p /content/drive/MyDrive/TCC/runs

# Fine-tuning a partir do best.pt da v3 (nao do zero)
BASE_MODEL = '/content/drive/MyDrive/TCC/runs/train-colab-v3/weights/best.pt'

!yolo task=detect mode=train \
  model={BASE_MODEL} \
  data={DATA_DIR}/data.yaml \
  epochs=40 \
  batch=16 \
  imgsz=640 \
  patience=15 \
  project=/content/drive/MyDrive/TCC/runs \
  name=train-colab-v4


## Validacao

In [ ]:
!yolo task=detect mode=val \
  model=/content/drive/MyDrive/TCC/runs/train-colab-v4/weights/best.pt \
  data={DATA_DIR}/data.yaml \
  split=test


## Comparacao no conjunto de teste real (antes vs depois)

Mede o modelo v2 (antes do fine-tuning) e o v3 (depois) nas mesmas 25 imagens reais reservadas - nunca vistas em nenhum treino. Essa e a comparacao que interessa para o TCC: mostra se o fine-tuning realmente melhorou a deteccao em video/imagem real, nao so no split de teste original (que vem da mesma distribuicao do treino).

In [ ]:
import yaml

real_test_yaml = f'{REAL_TEST_DIR}/data.yaml'
with open(real_test_yaml, 'w') as f:
    f.write(f'train: {REAL_TEST_DIR}/images\n')
    f.write(f'val: {REAL_TEST_DIR}/images\n')
    f.write('nc: 5\n')
    f.write("names: ['helmet', 'vest', 'gloves', 'boots', 'goggles']\n")

print('=== Modelo v2 (ANTES do fine-tuning) no conjunto real ===')
!yolo task=detect mode=val \
  model=/content/drive/MyDrive/TCC/runs/train-colab-v2/weights/best.pt \
  data={real_test_yaml}

print()
print('=== Modelo v3 (DEPOIS do fine-tuning) no conjunto real ===')
!yolo task=detect mode=val \
  model=/content/drive/MyDrive/TCC/runs/train-colab-v3/weights/best.pt \
  data={real_test_yaml}


## Comparacao no conjunto de teste real (v3 vs v4)

Mesma logica da comparacao v2/v3 anterior, agora medindo se adicionar o Construction-PPE (mais dados de gloves/goggles) melhorou a deteccao nas 25 imagens reais reservadas.

In [ ]:
print('=== Modelo v3 (ANTES do Construction-PPE) no conjunto real ===')
!yolo task=detect mode=val \
  model=/content/drive/MyDrive/TCC/runs/train-colab-v3/weights/best.pt \
  data={real_test_yaml}

print()
print('=== Modelo v4 (DEPOIS do Construction-PPE) no conjunto real ===')
!yolo task=detect mode=val \
  model=/content/drive/MyDrive/TCC/runs/train-colab-v4/weights/best.pt \
  data={real_test_yaml}
